# Static Accuracy Analysis

Loads the most recent `static_accuracy` run and inspects the EE position recorded by the camera vs the FK-predicted position. Volcaniarm is a 2-DOF planar arm so X is fixed mechanical bias (not an error metric); accuracy is measured in the Y-Z plane only.

Two complementary views of the same data:
1. **Vector residual** in the Y-Z plane (`dy, dz, dr_yz`) — biased by URDF mount placeholders, but variance is clean.
2. **Scalar Euclidean distance** `|EE − base|` from FK and from tag, compared as numbers — also biased by URDF mounts at any single goal, but easier to reason about and the bias is constant within a run.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    align_fk_to_tag, latest_run, list_runs, load_run,
)

TEST_NAME = 'static_accuracy'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:  {config.get("run_id")}')
print(f'status:  {config.get("status")}  failure_reason: {config.get("failure_reason")}')
print(f'goal:    y={config["goals"][0][0]}, z={config["goals"][0][1]}')
print(f'samples per visit: {config["samples_per_visit"]}')
print(f'tag rows: {len(tag)}, fk rows: {len(fk)}')

## Raw measurements (Y-Z only)

What the camera saw at the target pose, in `apriltag_marker_base` frame. X is dropped — Volcaniarm doesn't move in X, so any X variation is fixed bias plus measurement noise on a non-controllable axis.

In [ ]:
target = tag[tag['phase'] == 'target']
home = tag[tag['phase'] == 'home']

print('TARGET phase rows:')
print(target[['cycle', 'sample_idx', 'y', 'z']].to_string(index=False))
print('\nHOME phase rows (baseline):')
print(home[['cycle', 'sample_idx', 'y', 'z']].to_string(index=False))
print('\nFK predictions (volcaniarm_base_link → tip):')
print(fk[['cycle', 'target_idx', 'fk_y', 'fk_z']].to_string(index=False))

## Euclidean distance: |EE − base| from FK vs from tag

The most intuitive scalar comparison: how far is the EE from base, according to each measurement source? FK uses the link origins (`|right_arm_tip − volcaniarm_base|`); tag uses the AprilTag centres (`|apriltag_marker_ee − apriltag_marker_base|`). They differ by a constant offset per goal due to the URDF mount placeholders, so:
- Mean difference per goal = the URDF-bias-baked-into-this-pose.
- Stddev of either across samples = repeatability.

In [ ]:
# Full 3D distance for the tag (it lives in 3D regardless of arm DoF) and
# the FK chain (which has fk_x ≈ 0 for a planar arm but include it for
# completeness; it's a constant per pose).
tag_target = tag[tag['phase'] == 'target'].copy()
tag_target['d_tag_m'] = np.sqrt(
    tag_target['x'] ** 2 + tag_target['y'] ** 2 + tag_target['z'] ** 2)

fk_distances = fk.copy()
fk_distances['d_fk_m'] = np.sqrt(
    fk_distances['fk_x'] ** 2 + fk_distances['fk_y'] ** 2 + fk_distances['fk_z'] ** 2)

# Pair each target sample with the matching FK row by (cycle, target_idx)
tag_target['target_idx'] = tag_target['target_idx'].astype(int)
fk_distances['target_idx'] = fk_distances['target_idx'].astype(int)
dist_join = tag_target.merge(
    fk_distances[['cycle', 'target_idx', 'd_fk_m']],
    on=['cycle', 'target_idx'], how='left')
dist_join['d_diff_m'] = dist_join['d_tag_m'] - dist_join['d_fk_m']

print(dist_join[['cycle', 'sample_idx', 'd_tag_m', 'd_fk_m', 'd_diff_m']]
      .to_string(index=False))
print(f'\n|d_tag| mean: {dist_join["d_tag_m"].mean()*1000:.2f} mm  '
      f'std: {dist_join["d_tag_m"].std()*1000:.2f} mm   (repeatability of tag distance)')
print(f'|d_fk|  mean: {dist_join["d_fk_m"].mean()*1000:.2f} mm   '
      f'(constant per goal; FK is deterministic in joint angles)')
print(f'|d_diff| mean: {dist_join["d_diff_m"].mean()*1000:+.2f} mm  '
      f'std: {dist_join["d_diff_m"].std()*1000:.2f} mm')
print('\nMean of d_diff is the URDF mount placeholder bias for this goal '
      '(constant per pose).\nStd of d_diff is the measurement noise + arm '
      'repeatability (the metric you actually care about).')

## Per-sample variance, Y and Z

Stddev across samples within the target visit. Sub-mm here = stable detection. With a single sample this prints zeros — to get meaningful variance run with `samples_per_visit:=5` or higher.

In [ ]:
stats = target[['y', 'z']].agg(['mean', 'std']).T
stats.columns = ['mean_m', 'std_m']
stats['std_mm'] = stats['std_m'] * 1000
print(stats)

## 2D scatter: workspace Y-Z plane

Goal (red star), FK prediction (orange circle), and tag-measured position (blue x). FK is in `volcaniarm_base_link` frame; tag y/z is in `apriltag_marker_base` frame — separated by the URDF static offset (currently a placeholder, hence the visible gap).

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))

gy, gz = config['goals'][0][0], config['goals'][0][1]
ax.scatter([gy], [gz], marker='*', s=300, c='red', label='goal', zorder=5)
ax.scatter(fk['fk_y'], fk['fk_z'], marker='o', s=120, facecolors='none',
           edgecolors='orange', linewidths=2, label='FK', zorder=4)
ax.scatter(target['y'], target['z'], marker='x', s=80, c='steelblue',
           label='tag-measured', zorder=3)
ax.set_xlabel('y [m]')
ax.set_ylabel('z [m]')
ax.set_title(f'y-z workspace plane  ({config["run_id"]})')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend()
fig.tight_layout()

## Vector residuals (tag − FK), Y and Z only

Component-level view of the same residual. `dr_yz = sqrt(dy² + dz²)` is the in-plane residual magnitude. Same caveat as above — biased by URDF mount placeholders; variance is clean.

In [ ]:
merged = align_fk_to_tag(tag, fk)
merged['dr_yz'] = (merged['dy'] ** 2 + merged['dz'] ** 2) ** 0.5
print(merged[['cycle', 'sample_idx', 'dy', 'dz', 'dr_yz']].to_string(index=False))
print(f'\n|dr_yz| mean: {merged["dr_yz"].mean()*1000:.2f} mm')
print(f'|dr_yz| std:  {merged["dr_yz"].std()*1000:.2f} mm   (meaningful with multiple samples)')

## Next iterations to fill in

Once a run with multiple samples per visit is available:
1. Drift-within-visit time-series (settling sanity check) using `t_ros_ns`.
2. Per-cycle mean/std table once we have multiple cycles too.
3. Quaternion analysis: orientation residual between measured tag pose and the URDF-implied apriltag_marker_ee pose.